# Part C: Combine PRN2023 + GE2022 into the BN Table 5/6-style deliverable

Joins Part A's PRN2023-side BN support table (`N9_State Election_analysis.ipynb`) with
Part B's GE2022-side BN support table (`N9_General Election_analysis.ipynb`) via their
exported CSVs (`Data/bn_support_prn2023.csv` / `Data/bn_support_ge2022.csv`) — these
two notebooks don't share kernel state, so the numeric `bn_support` tables were
exported at the end of each rather than re-running either pipeline here.

Goal: for each of the 17 NS seats BN contested in PRN2023, show GE2022 vs PRN2023
Malay/Chinese support and the percentage-point change — the BN-equivalent of the
paper's Table 5/6 — then check the boss's working hypothesis (BN gained Chinese
support via PH transfers but lost Malay support to PN).

In [1]:
import pandas as pd
import numpy as np

prn2023 = pd.read_csv('Data/bn_support_prn2023.csv')
ge2022 = pd.read_csv('Data/bn_support_ge2022.csv')

print(prn2023.shape, ge2022.shape)  # expect (17, 5) each
assert len(prn2023) == 17 and len(ge2022) == 17
assert set(prn2023['dun']) == set(ge2022['dun']), "seat sets differ between the two years"

prn2023.head()

(17, 5) (17, 5)


,dun,party,n_streams,malay_support,chinese_support
0,N02 PERTANG,BN,26,0.552833,NaN
1,N03 SUNGAI LUI,BN,41,0.455297,NaN
2,N05 SERTING,BN,39,0.386563,NaN
3,N06 PALONG,BN,45,0.466409,NaN
4,N07 JERAM PADANG,BN,32,0.523923,NaN


## Join both years and compute percentage-point change

`malay_change_pp` / `chinese_change_pp` = PRN2023 support minus GE2022 support, in
percentage points. `NaN` propagates if either side is below the 20%-of-registered-
voters reporting threshold (can't compute a meaningful change without both values).

In [2]:
joined = prn2023.merge(
    ge2022,
    on='dun',
    suffixes=('_prn2023', '_ge2022'),
    validate='one_to_one',
)

joined['malay_change_pp'] = (joined['malay_support_prn2023'] - joined['malay_support_ge2022']) * 100
joined['chinese_change_pp'] = (joined['chinese_support_prn2023'] - joined['chinese_support_ge2022']) * 100

print(joined.shape)  # expect (17, 11)
joined[['dun', 'malay_support_ge2022', 'malay_support_prn2023', 'malay_change_pp',
        'chinese_support_ge2022', 'chinese_support_prn2023', 'chinese_change_pp']]

(17, 11)


,dun,malay_support_ge2022,malay_support_prn2023,malay_change_pp,chinese_support_ge2022,chinese_support_prn2023,chinese_change_pp
0,N02 PERTANG,0.616057,0.552833,-6.322478,0.059826,NaN,NaN
1,N03 SUNGAI LUI,0.596245,0.455297,-14.094782,NaN,NaN,NaN
2,N05 SERTING,0.567433,0.386563,-18.086992,NaN,NaN,NaN
3,N06 PALONG,0.527287,0.466409,-6.087801,NaN,NaN,NaN
4,N07 JERAM PADANG,0.567916,0.523923,-4.399321,NaN,NaN,NaN
5,N09 LENGGENG,0.368451,0.397466,2.901503,NaN,NaN,NaN
6,N15 JUASSEH,0.515289,0.443905,-7.138423,NaN,NaN,NaN
7,N16 SERI MENANTI,0.490088,0.525547,3.545896,NaN,NaN,NaN
8,N17 SENALING,0.565042,0.450750,-11.429222,NaN,NaN,NaN
9,N19 JOHOL,0.541707,0.534800,-0.690754,NaN,NaN,NaN


## Final seat-level output table

Same paper-style formatting as `bn_output`/`bn_output_ge2022` (percentages with
"N.A." below the 20% threshold), now with both years side by side plus the
percentage-point change. This is the seat-level (Table 6-style) half of the
deliverable.

In [3]:
def format_pct(value):
    return "N.A." if pd.isna(value) else f"{value * 100:.1f}%"

def format_pp(value):
    return "N.A." if pd.isna(value) else f"{value:+.1f}pp"

seat_split = joined['dun'].str.extract(r'^(N\d+)\s+(.*)$')
final_table = pd.DataFrame({
    'seat_code': seat_split[0],
    'seat_name': seat_split[1].str.title(),
    'malay_ge2022': joined['malay_support_ge2022'].apply(format_pct),
    'malay_prn2023': joined['malay_support_prn2023'].apply(format_pct),
    'malay_change': joined['malay_change_pp'].apply(format_pp),
    'chinese_ge2022': joined['chinese_support_ge2022'].apply(format_pct),
    'chinese_prn2023': joined['chinese_support_prn2023'].apply(format_pct),
    'chinese_change': joined['chinese_change_pp'].apply(format_pp),
})
assert final_table[['seat_code', 'seat_name']].isna().sum().sum() == 0, "seat code/name split failed for some row"

final_table = final_table.sort_values('seat_code').reset_index(drop=True)
print(final_table.shape)
final_table

(17, 8)


,seat_code,seat_name,malay_ge2022,malay_prn2023,malay_change,chinese_ge2022,chinese_prn2023,chinese_change
0,N02,Pertang,61.6%,55.3%,-6.3pp,6.0%,N.A.,N.A.
1,N03,Sungai Lui,59.6%,45.5%,-14.1pp,N.A.,N.A.,N.A.
2,N05,Serting,56.7%,38.7%,-18.1pp,N.A.,N.A.,N.A.
3,N06,Palong,52.7%,46.6%,-6.1pp,N.A.,N.A.,N.A.
4,N07,Jeram Padang,56.8%,52.4%,-4.4pp,N.A.,N.A.,N.A.
5,N09,Lenggeng,36.8%,39.7%,+2.9pp,N.A.,N.A.,N.A.
6,N15,Juasseh,51.5%,44.4%,-7.1pp,N.A.,N.A.,N.A.
7,N16,Seri Menanti,49.0%,52.6%,+3.5pp,N.A.,N.A.,N.A.
8,N17,Senaling,56.5%,45.1%,-11.4pp,N.A.,N.A.,N.A.
9,N19,Johol,54.2%,53.5%,-0.7pp,N.A.,N.A.,N.A.


## State-level summary (Table 5 replica)

The paper's actual Table 5 is a **per-state average** across a party's contested
seats (not per-seat like Table 6). This computes the NS-state BN row: mean Malay/
Chinese support in GE2022 vs PRN2023, averaged across the seats where each group
clears the 20% reporting threshold (mirrors how the paper handles Lobak/Mambau's
"N.A." Malay entries in Table 6 — excluded from the average, not treated as 0).

In [4]:
state_summary = pd.DataFrame([{
    'party': 'BN',
    'n_seats': len(joined),
    'malay_ge2022': joined['malay_support_ge2022'].mean(),
    'malay_prn2023': joined['malay_support_prn2023'].mean(),
    'chinese_ge2022': joined['chinese_support_ge2022'].mean(),
    'chinese_prn2023': joined['chinese_support_prn2023'].mean(),
    'n_seats_chinese_reportable': joined['chinese_support_ge2022'].notna().sum(),
}])
state_summary['malay_change_pp'] = (state_summary['malay_prn2023'] - state_summary['malay_ge2022']) * 100
state_summary['chinese_change_pp'] = (state_summary['chinese_prn2023'] - state_summary['chinese_ge2022']) * 100

for col in ['malay_ge2022', 'malay_prn2023', 'chinese_ge2022', 'chinese_prn2023']:
    state_summary[col] = state_summary[col].apply(format_pct)
for col in ['malay_change_pp', 'chinese_change_pp']:
    state_summary[col] = state_summary[col].apply(format_pp)

state_summary

,party,n_seats,malay_ge2022,malay_prn2023,chinese_ge2022,chinese_prn2023,n_seats_chinese_reportable,malay_change_pp,chinese_change_pp
0,BN,17,53.9%,46.1%,5.1%,99.0%,2,-7.8pp,+93.9pp


## Testing the boss's working hypothesis

Hypothesis: BN gained Chinese support (PH vote transfers) but lost Malay support (to
PN) between GE2022 and PRN2023. Classify each of the 17 seats against this pattern —
without pre-filtering, per the brief — then inspect which seats don't fit and why.

In [5]:
hyp = joined[['dun', 'malay_change_pp', 'chinese_change_pp']].copy()
hyp['malay_fell'] = hyp['malay_change_pp'] < 0
hyp['chinese_rose'] = hyp['chinese_change_pp'] > 0  # only defined where reportable both years

print(f"Malay support fell in {hyp['malay_fell'].sum()} of {len(hyp)} BN seats "
      f"(rose/flat in {(~hyp['malay_fell']).sum()}).")
print(f"Chinese support is reportable (>=20% of registered voters) in both years "
      f"for only {hyp['chinese_change_pp'].notna().sum()} of {len(hyp)} seats "
      f"— the hypothesis's Chinese-transfer half can only be directly tested there.")

hyp.sort_values('malay_change_pp')

Malay support fell in 14 of 17 BN seats (rose/flat in 3).
Chinese support is reportable (>=20% of registered voters) in both years for only 1 of 17 seats — the hypothesis's Chinese-transfer half can only be directly tested there.


,dun,malay_change_pp,chinese_change_pp,malay_fell,chinese_rose
16,N35 GEMENCHEH,-18.792029,94.784932,True,True
2,N05 SERTING,-18.086992,NaN,True,False
12,N28 KOTA,-17.579008,NaN,True,False
15,N34 GEMAS,-14.545568,NaN,True,False
1,N03 SUNGAI LUI,-14.094782,NaN,True,False
11,N27 RANTAU,-12.208823,NaN,True,False
8,N17 SENALING,-11.429222,NaN,True,False
10,N26 CHEMBONG,-7.605322,NaN,True,False
6,N15 JUASSEH,-7.138423,NaN,True,False
0,N02 PERTANG,-6.322478,NaN,True,False
